In [41]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import root_mean_squared_error, r2_score
import shap
import warnings
warnings.filterwarnings("ignore")
import matplotlib.cm as cm
%matplotlib inline

In [42]:
df_2020 = pd.read_csv('data/delta_table_2020_3x3.csv')
df_2021 = pd.read_csv('data/delta_table_2021_3x3.csv')

df_2021 = df_2021[df_2021['delta_years'] != 5]

In [43]:
df_2020.head()

,delta_years,system:index,B11,B12,B2,B3,B4,B5,B6,B7,...,B8_p0_p1,B11_p1_p1,B12_p1_p1,B2_p1_p1,B3_p1_p1,B4_p1_p1,B5_p1_p1,B6_p1_p1,B7_p1_p1,B8_p1_p1
0,3,2,2427.0,1737.0,1295.0,1608.0,1579.0,1899.0,2650.0,2975.0,...,3340.0,2324.0,1565.0,967.0,1162.0,1179.0,1584.0,2924.0,3301.0,3431.0
1,3,3,2431.0,1874.0,1327.0,1607.0,1769.0,2144.0,2683.0,2822.0,...,3431.0,2467.0,1860.0,1305.0,1567.0,1659.0,1903.0,2585.0,2891.0,2877.0
2,3,4,2311.0,1894.0,1334.0,1550.0,1615.0,1864.0,1944.0,2010.0,...,2877.0,2317.0,1821.0,1210.0,1312.0,1384.0,1508.0,2155.0,2360.0,2539.0
3,3,5,2225.0,1964.0,1119.0,1194.0,1220.0,1352.0,1382.0,1394.0,...,2539.0,2233.0,1750.0,1142.0,1270.0,1363.0,1569.0,2262.0,2541.0,2492.0
4,3,6,2291.0,1958.0,1192.0,1290.0,1376.0,1714.0,1666.0,1756.0,...,2492.0,2343.0,1635.0,1094.0,1330.0,1556.0,1909.0,2381.0,2617.0,2623.0


In [44]:
df_2020['B11'].mean(), df_2020['B11'].std()

(np.float64(1826.7826415779152), np.float64(537.6874636896489))

In [45]:
df_2020 = pd.read_csv('data/delta_table_2020_3x3_25_percent.csv')
df_2021 = pd.read_csv('data/delta_table_2021_3x3_25_percent.csv')

df_2021 = df_2021[df_2021['delta_years'] != 5]

In [46]:
df_2020.head()

,delta_years,system:index,B11,B12,B2,B3,B4,B5,B6,B7,...,B8_p0_p1,B11_p1_p1,B12_p1_p1,B2_p1_p1,B3_p1_p1,B4_p1_p1,B5_p1_p1,B6_p1_p1,B7_p1_p1,B8_p1_p1
0,3,2,2536.634024,1730.663483,1275.383544,1533.979055,1735.681885,1880.642249,2586.945175,2914.190451,...,3270.283588,2364.447599,1463.616847,929.698359,1189.060543,1287.463012,1621.907741,3109.356315,3284.600547,3369.742880
1,3,3,2476.608845,1926.383366,1372.716073,1839.809371,1716.913523,2083.146729,2563.203212,2928.372798,...,3369.742880,2804.467888,1910.036015,1228.653047,1491.529549,1538.185381,2014.262310,2531.359435,2912.994084,2865.287998
2,3,4,2296.778913,1787.368677,1246.511129,1620.723165,1710.256055,1928.525492,1846.029249,2050.091680,...,2865.287998,2408.293106,1713.532588,1095.866890,1336.382403,1461.019427,1547.504083,2035.839019,2476.994101,2401.820304
3,3,5,2252.293190,2074.600353,1142.726583,1267.801796,1228.660988,1386.346727,1474.932952,1142.365389,...,2401.820304,2167.909943,1630.194921,1280.070915,1338.856553,1440.536439,1717.165880,2676.054605,2245.835653,2355.831704
4,3,6,2232.295658,1876.914641,1162.803598,1452.508690,1265.034385,1829.029266,1418.262043,1698.169078,...,2355.831704,2266.098946,1527.015328,979.653499,1297.425994,1400.079086,1948.802271,2388.610593,2700.068303,2716.633140


In [47]:
df_2020['B11'].mean(), df_2020['B11'].std()

(np.float64(1826.7047055194805), np.float64(554.0103364915585))

In [48]:
df_2020 = df_2020.drop(columns=['year', "system:index", "image_count", ".geo", "geometry", "x", "y"])
df_2020 = df_2020.fillna(df_2020.mean())

df_2021 = df_2021.drop(columns=['year', "system:index", "image_count", ".geo", "geometry", "x", "y"])
df_2021 = df_2021.fillna(df_2021.mean())


In [49]:
df_2020['vegetation'] = df_2020[['tree_cover', 'cropland', 'grassland']].sum(axis=1)
df_2021['vegetation'] = df_2021[['tree_cover', 'cropland', 'grassland']].sum(axis=1)

target_labels = [
    'built_up',
    'vegetation',
    'water'
]

remaining_target = [
    'bare_sparse_vegetation',
    'tree_cover',
    'grassland',
    'cropland'
]


In [50]:
# --- Feature Engineering for df_2020 ---
# Vegetation Indices
df_2020['NDVI'] = (df_2020['B8'] - df_2020['B4']) / (df_2020['B8'] + df_2020['B4'] + 1e-8)
# df_2020['EVI'] = 2.5 * ((df_2020['B8'] - df_2020['B4']) / (df_2020['B8'] + 6 * df_2020['B4'] - 7.5 * df_2020['B2'] + 1 + 1e-8))
df_2020['EVI2'] = 2.5 * ((df_2020['B8'] - df_2020['B4']) / (df_2020['B8'] + 2.4 * df_2020['B4'] + 1 + 1e-8))
df_2020['SAVI'] = ((df_2020['B8'] - df_2020['B4']) * 1.5) / (df_2020['B8'] + df_2020['B4'] + 0.5 + 1e-8)
# df_2020['NDRE'] = (df_2020['B8'] - df_2020['B5']) / (df_2020['B8'] + df_2020['B5'] + 1e-8)
# df_2020['GNDVI'] = (df_2020['B8'] - df_2020['B3']) / (df_2020['B8'] + df_2020['B3'] + 1e-8)

# Urban & Built-up Indices
df_2020['NDBI'] = (df_2020['B11'] - df_2020['B8']) / (df_2020['B11'] + df_2020['B8'] + 1e-8)

# Water & Moisture Indices
df_2020['NDWI'] = (df_2020['B3'] - df_2020['B8']) / (df_2020['B3'] + df_2020['B8'] + 1e-8)
df_2020['MNDWI'] = (df_2020['B3'] - df_2020['B11']) / (df_2020['B3'] + df_2020['B11'] + 1e-8)


# --- Feature Engineering for df_2021 ---
# Vegetation Indices
df_2021['NDVI'] = (df_2021['B8'] - df_2021['B4']) / (df_2021['B8'] + df_2021['B4'] + 1e-8)
# df_2021['EVI'] = 2.5 * ((df_2021['B8'] - df_2021['B4']) / (df_2021['B8'] + 6 * df_2021['B4'] - 7.5 * df_2021['B2'] + 1 + 1e-8))
df_2021['EVI2'] = 2.5 * ((df_2021['B8'] - df_2021['B4']) / (df_2021['B8'] + 2.4 * df_2021['B4'] + 1 + 1e-8))
df_2021['SAVI'] = ((df_2021['B8'] - df_2021['B4']) * 1.5) / (df_2021['B8'] + df_2021['B4'] + 0.5 + 1e-8)
# df_2021['NDRE'] = (df_2021['B8'] - df_2021['B5']) / (df_2021['B8'] + df_2021['B5'] + 1e-8)
# df_2021['GNDVI'] = (df_2021['B8'] - df_2021['B3']) / (df_2021['B8'] + df_2021['B3'] + 1e-8)

# Urban & Built-up Indices
df_2021['NDBI'] = (df_2021['B11'] - df_2021['B8']) / (df_2021['B11'] + df_2021['B8'] + 1e-8)

# Water & Moisture Indices
df_2021['NDWI'] = (df_2021['B3'] - df_2021['B8']) / (df_2021['B3'] + df_2021['B8'] + 1e-8)
df_2021['MNDWI'] = (df_2021['B3'] - df_2021['B11']) / (df_2021['B3'] + df_2021['B11'] + 1e-8)

In [51]:
X_train = df_2020.drop(columns=target_labels + remaining_target)
X_train = X_train.drop(columns=['B4', 'B7', 'B8', 'NDBI'])
y_train = df_2020[target_labels]

X_test = df_2021.drop(columns=target_labels + remaining_target)
X_test = X_test.drop(columns=['B4', 'B7', 'B8', 'NDBI'])
y_test = df_2021[target_labels]

In [52]:
X_train.columns, y_train.columns

(Index(['delta_years', 'B11', 'B12', 'B2', 'B3', 'B5', 'B6', 'B11_m1_m1',
        'B12_m1_m1', 'B2_m1_m1', 'B3_m1_m1', 'B4_m1_m1', 'B5_m1_m1', 'B6_m1_m1',
        'B7_m1_m1', 'B8_m1_m1', 'B11_p0_m1', 'B12_p0_m1', 'B2_p0_m1',
        'B3_p0_m1', 'B4_p0_m1', 'B5_p0_m1', 'B6_p0_m1', 'B7_p0_m1', 'B8_p0_m1',
        'B11_p1_m1', 'B12_p1_m1', 'B2_p1_m1', 'B3_p1_m1', 'B4_p1_m1',
        'B5_p1_m1', 'B6_p1_m1', 'B7_p1_m1', 'B8_p1_m1', 'B11_m1_p0',
        'B12_m1_p0', 'B2_m1_p0', 'B3_m1_p0', 'B4_m1_p0', 'B5_m1_p0', 'B6_m1_p0',
        'B7_m1_p0', 'B8_m1_p0', 'B11_p1_p0', 'B12_p1_p0', 'B2_p1_p0',
        'B3_p1_p0', 'B4_p1_p0', 'B5_p1_p0', 'B6_p1_p0', 'B7_p1_p0', 'B8_p1_p0',
        'B11_m1_p1', 'B12_m1_p1', 'B2_m1_p1', 'B3_m1_p1', 'B4_m1_p1',
        'B5_m1_p1', 'B6_m1_p1', 'B7_m1_p1', 'B8_m1_p1', 'B11_p0_p1',
        'B12_p0_p1', 'B2_p0_p1', 'B3_p0_p1', 'B4_p0_p1', 'B5_p0_p1', 'B6_p0_p1',
        'B7_p0_p1', 'B8_p0_p1', 'B11_p1_p1', 'B12_p1_p1', 'B2_p1_p1',
        'B3_p1_p1', 'B4_p1_p1', 'B5

In [53]:
import pickle

with open('models/XGBoost_delta.pkl', 'rb') as f:
    loaded_models = pickle.load(f)

best_model = loaded_models

y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)

In [54]:
X_test

,delta_years,B11,B12,B2,B3,B5,B6,B11_m1_m1,B12_m1_m1,B2_m1_m1,...,B4_p1_p1,B5_p1_p1,B6_p1_p1,B7_p1_p1,B8_p1_p1,NDVI,EVI2,SAVI,NDWI,MNDWI
0,3,2478.627548,1656.500969,1134.531670,1432.534310,1846.944240,2473.319077,2199.150590,1781.933770,553.551464,...,759.266765,1367.739696,2500.835076,3255.646694,3266.817266,0.361162,0.623800,0.541678,-0.326527,-0.267464
1,3,2386.978382,1998.679813,1401.633355,1403.885826,2296.638352,2538.170659,2092.737684,1534.319118,1028.113700,...,1132.313634,1474.493845,2366.826823,2661.977268,2372.910500,0.266478,0.440105,0.399667,-0.282589,-0.259332
2,3,2373.056558,2016.050667,1153.331783,1515.026415,1636.228935,1648.461330,2106.452155,2110.127890,1357.172141,...,1313.864875,1387.849892,1845.646060,1987.709803,2380.903843,0.128883,0.200120,0.193297,-0.121111,-0.220682
3,3,2017.541159,1994.490023,1127.085893,1130.320781,1388.111739,1429.032616,2201.990117,1526.286053,1240.831819,...,1233.265694,1416.585084,1933.793059,2338.684086,2115.721655,0.140022,0.218456,0.209989,-0.089951,-0.281849
4,3,2327.295990,2152.444420,1073.014590,1268.794244,1483.305158,1600.896698,2080.225897,1986.714368,1075.028705,...,1390.944719,1885.507013,2282.486982,2534.000911,2737.407219,-0.009031,-0.013229,-0.013544,-0.053043,-0.294348
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2286810,0,1708.798972,1439.115857,647.283990,772.018755,1120.549138,1802.368273,1055.256420,887.368437,97.594490,...,812.846920,1577.455412,2298.220019,2258.685835,2798.430081,0.383610,0.669786,0.575310,-0.419282,-0.377609
2286811,0,1543.737718,1142.294045,507.753163,550.316145,1181.532808,1735.449998,1086.520385,883.858670,198.913447,...,1114.989713,1537.175792,1727.938807,2205.384536,1849.827058,0.324165,0.549997,0.486150,-0.501971,-0.474401
2286812,0,1622.236978,1095.475411,390.935938,614.316366,628.927831,1268.453062,1276.962858,612.971965,222.812074,...,1101.996261,1240.804600,2055.749977,2845.780885,3141.430900,0.443967,0.798677,0.665794,-0.426203,-0.450658
2286813,0,1100.717000,1071.744850,157.828555,490.993887,601.534740,1459.370779,1065.474976,783.517096,197.426767,...,636.325391,1295.166143,2635.843228,2743.340646,2898.394554,0.607358,1.190547,0.910797,-0.513969,-0.383061


In [55]:
X_test['delta_years'].unique()

array([3, 1, 4, 2, 0])

In [56]:
train_rmse = root_mean_squared_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

test_rmse = root_mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f"Train R2: {train_r2:.4f} | Test R2: {test_r2:.4f}")
print(f"Train RMSE: {train_rmse:.4f} | Test RMSE: {test_rmse:.4f}\n")

Train R2: 0.8237 | Test R2: 0.8070
Train RMSE: 0.1467 | Test RMSE: 0.1575



In [57]:
loaded_models

,estimator,"XGBRegressor(...ree=None, ...)"
,n_jobs,-1
,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.6983814307724542
,device,'cuda'
,early_stopping_rounds,None
